# Génération massive de synonymes cliniques — Phase 2 (API Batch Mistral)

## Contexte

Phase 2 du chantier d'enrichissement du CSV avec des formulations cliniques générées par LLM. Phase 1 (test sur 3 codes témoins) a validé le prompt — cf notebook `2026-06-07_test_generation_synonymes.ipynb`.

**Ce notebook génère les synonymes pour les 15 978 codes feuilles** via l'API Batch Mistral (–50 % vs API standard).

## Workflow d'utilisation

1. **Exécuter cellules 2 à 9** — setup (imports, config, secrets, chargements, prompts, batches construits)
2. **Exécuter cellule 10 (mini-batch de 10 codes)** — valide la mécanique de bout en bout et mesure le coût réel. Durée ~5-30 min selon la file d'attente Mistral.
3. **Vérifier les résultats du mini-batch + coût extrapolé**.
4. **Si OK : exécuter cellule 12 (4 batches complets)**. ⚠ Coût ~50 € — Durée 4-96 h.
5. **Exécuter cellules 13 (consolidation) et 14 (récap final)**.

## Reprise après interruption

Chaque batch terminé écrit son fichier dans `outputs/llm_synonymes/batches/`. Si la cellule 12 est ré-exécutée, les batches déjà téléchargés sont **skippés**.

## Pré-requis

- `uv sync --extra llm` (déjà fait pour la phase 1)
- `config/secrets.yaml` rempli avec la clé Mistral (déjà fait pour la phase 1)
- Bibliothèque `outputs/cards_library/` et fichier `_index.csv` à jour


In [ ]:
from __future__ import annotations

import json
import random
import re
import time
from pathlib import Path

import polars as pl
import yaml
from mistralai.client import Mistral

In [ ]:
# Détection robuste de la racine projet (chemins indépendants du cwd du kernel).
def _find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / "pyproject.toml").is_file():
            return p
        p = p.parent
    raise RuntimeError("pyproject.toml introuvable depuis cwd")


PROJECT_ROOT = _find_project_root()

# Chemins
CARDS_DIR = PROJECT_ROOT / "outputs/cards_library"
INDEX_PATH = CARDS_DIR / "_index.csv"
CEPIDC_PATH = PROJECT_ROOT / "data/CIM_CEPIDC_2015/CepiDc_Dictionnaire2015.csv"
SECRETS_PATH = PROJECT_ROOT / "config/secrets.yaml"

# Sorties
OUTPUT_DIR = PROJECT_ROOT / "outputs/llm_synonymes"
BATCH_FILES_DIR = OUTPUT_DIR / "batches"
CEPIDC_EXAMPLES_PATH = OUTPUT_DIR / "cepidc_examples_used.json"
CONSOLIDATED_OUTPUT = OUTPUT_DIR / "synonymes_consolide.jsonl"

# LLM (identique à phase 1)
MODEL = "mistral-large-latest"
TEMPERATURE = 0.5
MAX_TOKENS = 1500
N_TARGET = 20
MAX_CEPIDC_EXAMPLES = 5

# Batch
BATCH_SIZE = 5000
BATCH_TIMEOUT_HOURS = 24
POLL_INTERVAL_S = 30
POLL_VERBOSE_INTERVAL_S = 300  # print toutes les 5 min si pas de changement de status
SEED = 42

# Cohérence avec phase 1 pour les 3 codes témoins (réutilisés en spot-check)
CODES_TEMOINS = ["A18.1", "J18.8", "R51"]

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"INDEX_PATH   : {INDEX_PATH.is_file()} → {INDEX_PATH}")
print(f"CEPIDC_PATH  : {CEPIDC_PATH.is_file()} → {CEPIDC_PATH}")

In [ ]:
# Setup secrets + client Mistral.
if not SECRETS_PATH.is_file():
    raise RuntimeError(
        f"Fichier de configuration manquant : {SECRETS_PATH}.\n"
        f"Copier config/secrets.yaml.example → config/secrets.yaml et y mettre la clé Mistral."
    )

with SECRETS_PATH.open(encoding="utf-8") as f:
    secrets = yaml.safe_load(f) or {}

api_key = (secrets.get("mistral") or {}).get("api_key")
if not api_key or api_key == "your-mistral-api-key-here":
    raise RuntimeError("Clé mistral.api_key manquante ou non remplie dans config/secrets.yaml.")

# Timeout généreux (10 min) : nécessaire car les batches de 5 000 codes
# pèsent ~45 Mo en upload. Le défaut httpx (5 s) explose en WriteTimeout.
client = Mistral(api_key=api_key, timeout_ms=600_000)
print(f"Client Mistral prêt — modèle : {MODEL}, timeout=10 min")

In [ ]:
# Chargement de la liste complète des codes feuilles depuis _index.csv.
# Tri par chapitre romain (I → XXII) puis par code.

ROMAN_TO_INT = {
    "I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6,
    "VII": 7, "VIII": 8, "IX": 9, "X": 10, "XI": 11,
    "XII": 12, "XIII": 13, "XIV": 14, "XV": 15, "XVI": 16,
    "XVII": 17, "XVIII": 18, "XIX": 19, "XX": 20, "XXI": 21,
    "XXII": 22,
}

index_df = (
    pl.read_csv(INDEX_PATH)
    .with_columns(
        pl.col("chapter")
        .replace_strict(ROMAN_TO_INT, default=-1)
        .alias("chap_num")
    )
    .sort("chap_num", "code")
)
print(f"Total codes feuilles : {index_df.height:,}")
print()
print("Distribution par chapitre :")
print(index_df.group_by("chapter").len().sort("len", descending=True).head(10))

# Mapping pour la consolidation : code → libellé (depuis _index.csv canonique)
code_to_libelle = dict(zip(index_df["code"].to_list(), index_df["libelle"].to_list(), strict=True))
codes_sorted = index_df["code"].to_list()
print()
print(f"Premier code : {codes_sorted[0]}")
print(f"Dernier code : {codes_sorted[-1]}")

In [ ]:
# Chargement et préparation du dictionnaire CepiDc.
# - Séparateur `;`, encodage utf-8
# - Filtrage : Icd1 non nul
# - Conversion `A181` → `A18.1` (point après le 3e caractère pour les codes >= 4 car)


def icd_to_dotted(icd: str | None) -> str | None:
    """A181 → A18.1, R51 → R51, X44 → X44."""
    if icd is None:
        return None
    if len(icd) <= 3:
        return icd
    return f"{icd[:3]}.{icd[3:]}"


cepidc = (
    pl.read_csv(CEPIDC_PATH, separator=";", infer_schema_length=10_000)
    .filter(pl.col("Icd1").is_not_null())
    .with_columns(
        pl.col("Icd1")
        .map_elements(icd_to_dotted, return_dtype=pl.String)
        .alias("code_dotted")
    )
    .select("code_dotted", "DiagnosisText")
)
print(f"CepiDc chargé : {cepidc.height:,} lignes / {cepidc['code_dotted'].n_unique():,} codes uniques")
for code in CODES_TEMOINS:
    n = cepidc.filter(pl.col("code_dotted") == code).height
    print(f"  {code:6s} : {n:3d} formulations CepiDc")

In [ ]:
# Prompts et helpers RÉUTILISÉS TEL QUELS depuis le notebook phase 1
# (cf scripts/explore/2026-06-07_test_generation_synonymes.ipynb).
# Si tu modifies les prompts, propage la modification ici ET dans phase 1.

# Helper : chargement d'une fiche markdown depuis cards_library/.
# Les fiches sont rangées dans des sous-dossiers par chapitre romain.


def load_card(code: str, cards_dir: Path = CARDS_DIR) -> str | None:
    """Cherche `<code>.md` dans les sous-dossiers de `cards_dir`. None si absent."""
    for chap_dir in cards_dir.iterdir():
        if not chap_dir.is_dir():
            continue
        target = chap_dir / f"{code}.md"
        if target.is_file():
            return target.read_text(encoding="utf-8")
    return None


def extract_libelle_from_card(card: str) -> str:
    """Extrait le libellé après `# CODE — ` dans le titre de la fiche."""
    m = re.search(r"^# \S+ — (.+)$", card, re.MULTILINE)
    return m.group(1) if m else ""


def get_cepidc_examples(
    code: str,
    cepidc_df: pl.DataFrame,
    max_n: int = MAX_CEPIDC_EXAMPLES,
    rng: random.Random | None = None,
) -> list[str]:
    """Renvoie jusqu'à `max_n` DiagnosisText pour `code` ; sample reproductible si plus."""
    sub = cepidc_df.filter(pl.col("code_dotted") == code)
    texts = sub["DiagnosisText"].drop_nulls().to_list()
    if len(texts) > max_n:
        rng = rng or random.Random(SEED)
        texts = rng.sample(texts, max_n)
    return texts


# System prompt : rôle et mission du LLM.

SYSTEM_PROMPT = """Tu es un médecin codeur français expérimenté, spécialiste de la CIM-10
et de la pratique clinique hospitalière en France. Tu as lu des
milliers de comptes-rendus d'hospitalisation (CRH) rédigés par des
cliniciens et tu connais parfaitement le langage télégraphique, les
abréviations et les raccourcis qu'ils utilisent dans leur pratique
quotidienne.

# Contexte : le codage CIM-10

Le codage CIM-10 consiste à identifier dans un document médical (CRH,
notes d'évolution, comptes-rendus opératoires) les maladies,
symptômes, états cliniques et actes documentés par les médecins, et à
leur attribuer des codes standardisés. C'est une tâche de
standardisation d'information, pas une tâche de décision médicale :
on traduit ce que le clinicien a écrit en codes normalisés.

# Ta mission : la tâche inverse

Ta mission est de réaliser la tâche inverse : à partir d'un code
CIM-10 et de sa fiche descriptive officielle, générer les formulations
cliniques que des médecins auraient pu écrire dans un CRH pour aboutir
à ce code. Tu pars donc du concept standardisé et tu retournes vers la
diversité réelle de l'écriture médicale.

Ces formulations serviront à entraîner un outil d'aide au codage
automatique : il est essentiel qu'elles soient à la fois réalistes
(elles existent dans des CRH réels) et discriminantes (elles
permettent d'identifier le code).

# La nature variable des codes CIM-10

Les codes CIM-10 n'ont pas tous la même précision clinique. Certains
désignent une entité nette et unique (ex. I10, hypertension artérielle
essentielle). D'autres, en particulier les sous-catégories terminées
par .8 (« autres formes précisées »), regroupent plusieurs entités
hétérogènes mais identifiées sous un même libellé (ex. B17.8, « autres
hépatites virales aiguës précisées »). Le libellé officiel d'un code
ne suffit donc pas toujours à savoir quelle pathologie réelle décrire.

Cas particulier des codes « sans précision » (le plus souvent terminés
par .9) : ces codes ne sont employés que lorsque le dossier ne
contenait pas l'information détaillée qui aurait permis de choisir un
code plus spécifique. Leur présence est donc un signal positif
d'incertitude : on sait qu'il y a une pathologie de cette catégorie,
mais on ignore volontairement son détail.

En conséquence, quand tu génères des formulations pour un tel code,
reste délibérément vague. Ajouter une précision que le code ne porte
pas serait une erreur, car cela contredirait la raison même pour
laquelle ce code « sans précision » a été retenu plutôt qu'un code
plus fin. À l'inverse, un détail clinique inventé devrait toujours
justifier un code plus spécifique : son absence ici est intentionnelle.

Cas particulier des cancers : la CIM-10 a pour eux une approche
essentiellement anatomique. Le niveau de détail du code par rapport à
la catégorie est donc principalement une précision anatomique.

# Utilisation de la fiche descriptive

Pour t'aider, la fiche descriptive du code contient des informations
complémentaires issues de la classification et de thésaurus médicaux :

- **Synonymes et inclusions** : formulations cliniques alternatives
  et sous-types couverts par le code. Utilise-les pour choisir un
  vocabulaire naturel et varié, sans recopier le libellé officiel.
- **Exclusions** : entités voisines qui ne relèvent pas de ce code.
  Elles délimitent le périmètre. N'oriente jamais tes formulations
  vers une entité exclue.

Reste strictement dans le périmètre du code : reformule librement à
l'intérieur de ce que couvrent libellé + inclusions + synonymes, mais
n'en sors pas. Pour les codes « sans précision », ce périmètre est
volontairement large et flou — respecte ce flou."""

# User prompt : construction paramétrée.
# Bloc CepiDc varie selon présence/absence d'exemples.


_CEPIDC_WITH_EXAMPLES = """## Formulations existantes pour ce code (à NE PAS reproduire)

Les formulations suivantes sont déjà connues pour ce code. Elles te donnent le style attendu mais tu dois générer des formulations **complémentaires**, pas les reproduire :

{LIST}

Génère des formulations qui couvrent d'autres facettes du code, en gardant le même style télégraphique et clinique."""

_CEPIDC_WITHOUT_EXAMPLES = """## Formulations existantes pour ce code

Aucune formulation de référence n'est fournie pour ce code. Appuie-toi sur la fiche descriptive ci-dessus et sur ta connaissance de la pratique clinique française pour générer les formulations."""


def build_user_prompt(
    code: str,
    libelle: str,
    fiche: str,
    cepidc_examples: list[str],
    n_target: int = N_TARGET,
) -> str:
    if cepidc_examples:
        bloc = _CEPIDC_WITH_EXAMPLES.format(
            LIST="\n".join(f"- {e}" for e in cepidc_examples)
        )
    else:
        bloc = _CEPIDC_WITHOUT_EXAMPLES
    return f"""# Mission

Génère des formulations cliniques distinctes qui pourraient être effectivement présentes dans un compte-rendu d'hospitalisation français pour le code CIM-10 ci-dessous.

# Critères de validité

Une formulation est valide si et seulement si elle satisfait les 4 critères suivants :

1. **Réalisme CRH** : tu pourrais l'avoir lue telle quelle dans un CRH français récent rédigé par un clinicien (pas par un codeur ni un archiviste).

2. **Diversité de longueur** : génère un mélange équilibré de
   formulations courtes (1-3 mots, environ la moitié) et de
   formulations plus détaillées (4-8 mots, environ la moitié).
   Aucune formulation ne dépasse 8 mots.
3. **Vocabulaire clinique uniquement** : pas de termes spécifiques au vocabulaire de codage CIM-10 comme "sans précision", "SAI", "non classé ailleurs", "nca", "siège non précisé", etc. Ces mots n'apparaissent jamais dans un CRH écrit par un clinicien.

4. **Discrimination tolérante** : la formulation doit permettre d'identifier ce code spécifique. Une ambiguïté est acceptable si elle est typique de la pratique clinique réelle (par exemple, "pneumopathie" peut renvoyer à plusieurs codes, mais c'est le mot réellement utilisé en pratique).

# Type de formulations attendues

Variété attendue, classée du plus court au plus détaillé :

- **Abréviations courantes** : IDM, BPCO, AVC, BK, SCA, OAP, PTH, SARM, etc.
- **Formulations télégraphiques** : structure médicale rapide, sans articles ni mots de liaison superflus (ex. "TB rénale", "pyélo bilatérale", "IDM antéro-septal").
- **Variations idiomatiques** : différentes manières de dire la même chose en pratique clinique (ex. "tuberculose urogénitale", "tuberculose génito-urinaire", "TB génito-urinaire").
- **Formulations longues médicalement correctes** : 5 à 8 mots,
  attendues pour environ la moitié des générations. Exemples :
  "infection pleuro-pulmonaire à germes atypiques", "pyélonéphrite
  tuberculeuse chronique bilatérale", "pneumonie communautaire à
  germe non identifié", "céphalée chronique d'allure tensionnelle".
# Code à traiter

**Code** : {code}
**Libellé officiel** : {libelle}

## Fiche descriptive complète

{fiche}

{bloc}

# Format de sortie attendu

Réponds UNIQUEMENT avec un objet JSON valide contenant deux clés :

- `"code"` : le code CIM-10 traité
- `"formulations"` : une liste de strings, chaque string étant une formulation clinique distincte

```json
{{
  "code": "{code}",
  "formulations": ["formulation 1", "formulation 2", "formulation 3"]
}}
```

Aucun commentaire avant ou après le JSON. Aucun markdown. Juste le JSON.

# Nombre cible

Génère {n_target} formulations distinctes. Si tu ne peux pas en générer {n_target} crédibles (code rare, code très spécifique, peu de matière dans la fiche), génère moins. Mieux vaut 10 formulations crédibles que 20 dont 10 sont inventées.

Si une formulation t'apparaît douteuse (tu hésites à l'avoir vraiment lue dans un CRH), ne l'inclus pas."""

In [ ]:
# Découpage en batches de BATCH_SIZE codes.
chunks = [codes_sorted[i:i + BATCH_SIZE] for i in range(0, len(codes_sorted), BATCH_SIZE)]
print(f"Nombre de batches : {len(chunks)}")
for i, ch in enumerate(chunks, 1):
    print(f"  batch {i} : {len(ch):,} codes  ({ch[0]} → {ch[-1]})")

In [ ]:
# Helper : construction du fichier JSONL des requêtes pour un batch + 
# persistance du mapping code → cepidc_examples utilisés (pour la consolidation).
#
# Format Mistral Batch API : une ligne par requête, avec `custom_id` (=code)
# et `body` (messages, max_tokens, temperature, response_format).


def build_batch_jsonl(
    codes_chunk: list[str],
    batch_num: int,
    cepidc_df: pl.DataFrame,
    code_examples_map: dict[str, list[str]],
) -> Path:
    """Construit le fichier JSONL des requêtes et l'écrit sur disque.

    Met à jour `code_examples_map` (passé par référence) avec les exemples
    CepiDc utilisés pour chaque code. Cette persistance permet à la
    consolidation de reconstituer le champ `cepidc_examples` sans
    régénérer (cf décision : stockage explicite plutôt que reproductible
    via seed).
    """
    BATCH_FILES_DIR.mkdir(parents=True, exist_ok=True)
    requests_path = BATCH_FILES_DIR / f"batch_{batch_num}_requests.jsonl"
    # rng par batch : seed déterministe = (SEED, batch_num) pour traçabilité
    rng = random.Random(SEED + batch_num)
    n_missing_card = 0
    n_written = 0
    with requests_path.open("w", encoding="utf-8") as f:
        for code in codes_chunk:
            card = load_card(code)
            if card is None:
                n_missing_card += 1
                continue
            libelle = code_to_libelle.get(code) or extract_libelle_from_card(card)
            examples = get_cepidc_examples(code, cepidc_df, rng=rng)
            code_examples_map[code] = examples
            user_prompt = build_user_prompt(code, libelle, card, examples, N_TARGET)
            req = {
                "custom_id": code,
                "body": {
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt},
                    ],
                    "max_tokens": MAX_TOKENS,
                    "temperature": TEMPERATURE,
                    "response_format": {"type": "json_object"},
                },
            }
            f.write(json.dumps(req, ensure_ascii=False) + "\n")
            n_written += 1
    print(f"  batch {batch_num} : {n_written} requêtes écrites dans {requests_path.name}")
    if n_missing_card > 0:
        print(f"     ⚠ {n_missing_card} codes ignorés (fiche introuvable)")
    return requests_path

In [ ]:
# === MINI-BATCH (10 codes) — sécurité avant lancement complet ===
# À exécuter MANUELLEMENT, avant la cellule 12. Coût ~0,03 €.
#
# Valide :
#   - la signature exacte de client.files.upload / batch.jobs.create / download
#   - le format JSONL retourné par Mistral
#   - le coût empirique par code (extrapolable à 16 000)
#
# Si le mini-batch réussit : on peut lancer la cellule 12 en confiance.

import os

# On utilise un sous-ensemble du premier batch.
mini_codes = chunks[0][:10]

# On stocke les exemples CepiDc utilisés dans un dict en RAM (puis sauvé en JSON).
code_to_cepidc_examples_mini: dict[str, list[str]] = {}
mini_requests_path = build_batch_jsonl(
    mini_codes, batch_num=0, cepidc_df=cepidc,
    code_examples_map=code_to_cepidc_examples_mini,
)

# Upload → create → poll → download
with open(mini_requests_path, "rb") as f:
    content = f.read()
print(f"Upload du mini-batch ({len(content):,} bytes)...")
uploaded = client.files.upload(
    file={"file_name": mini_requests_path.name, "content": content},
    purpose="batch",
)
print(f"  uploaded.id = {uploaded.id}")

print("Création du job batch...")
job = client.batch.jobs.create(
    endpoint="/v1/chat/completions",
    input_files=[uploaded.id],
    model=MODEL,
    timeout_hours=BATCH_TIMEOUT_HOURS,
    metadata={"batch_num": "0", "phase": "mini"},
)
print(f"  job.id = {job.id}")

# Polling (verbeux espacé)
print("Polling (toutes les 5 min ou sur changement de status)...")
last_print = 0.0
last_status = None
t0 = time.time()
while True:
    job = client.batch.jobs.get(job_id=job.id)
    now = time.time()
    if job.status != last_status or now - last_print > POLL_VERBOSE_INTERVAL_S:
        print(
            f"    [{time.strftime('%H:%M:%S')}] status={job.status}  "
            f"completed={job.completed_requests}/{job.total_requests}"
        )
        last_status = job.status
        last_print = now
    if job.status in ("SUCCESS", "FAILED", "TIMEOUT_EXCEEDED", "CANCELLED"):
        break
    if now - t0 > BATCH_TIMEOUT_HOURS * 3600:
        raise RuntimeError(f"Timeout polling après {BATCH_TIMEOUT_HOURS}h")
    time.sleep(POLL_INTERVAL_S)

if job.status != "SUCCESS":
    raise RuntimeError(f"Mini-batch terminé avec status={job.status}")

# Download
mini_result_path = BATCH_FILES_DIR / "batch_0_results.jsonl"
print(f"Téléchargement des résultats vers {mini_result_path.name}...")
resp = client.files.download(file_id=job.output_file)
if hasattr(resp, "iter_bytes"):
    with open(mini_result_path, "wb") as f:
        for chunk in resp.iter_bytes():
            f.write(chunk)
elif hasattr(resp, "read"):
    mini_result_path.write_bytes(resp.read())
else:
    mini_result_path.write_bytes(bytes(resp))

# Affichage des résultats + coût empirique
n_ok = n_err = 0
tot_in = tot_out = 0
with mini_result_path.open() as f:
    for line in f:
        rec = json.loads(line)
        if rec.get("error") or rec["response"]["status_code"] != 200:
            n_err += 1
            continue
        n_ok += 1
        usage = rec["response"]["body"]["usage"]
        tot_in += usage["prompt_tokens"]
        tot_out += usage["completion_tokens"]
        content = rec["response"]["body"]["choices"][0]["message"]["content"]
        try:
            parsed = json.loads(content)
            print(f"  {rec['custom_id']:8s} → {len(parsed.get('formulations', []))} formulations")
        except json.JSONDecodeError:
            print(f"  {rec['custom_id']:8s} → JSON invalide")

cout_batch = tot_in / 1_000_000 * 1 + tot_out / 1_000_000 * 3  # tarif batch -50%
extrapol = cout_batch * (len(codes_sorted) / max(n_ok, 1))
print(f"\n=== Mini-batch terminé ===")
print(f"  Succès : {n_ok}/{len(mini_codes)}")
print(f"  Erreurs: {n_err}")
print(f"  Tokens : {tot_in:,} in  /  {tot_out:,} out")
print(f"  Coût mini-batch : ~{cout_batch:.4f} €")
print(f"  Coût extrapolé sur {len(codes_sorted):,} codes : ~{extrapol:.2f} €")
print(f"\n→ Si tout est OK, exécuter la cellule 12 (4 batches complets).")

In [ ]:
# Helper : exécution d'un batch (upload, create, polling espacé, download).
# Reprise après interruption : skip si fichier résultat déjà présent.


def run_batch(
    codes_chunk: list[str],
    batch_num: int,
    cepidc_df: pl.DataFrame,
    code_examples_map: dict[str, list[str]],
) -> Path:
    """Exécute un batch complet. Retourne le chemin du résultat JSONL."""
    result_path = BATCH_FILES_DIR / f"batch_{batch_num}_results.jsonl"
    if result_path.is_file():
        print(f"=== batch {batch_num} : déjà terminé ({result_path.name}) — skip ===")
        # Reconstituer code_examples_map à partir du JSONL des requêtes
        requests_path = BATCH_FILES_DIR / f"batch_{batch_num}_requests.jsonl"
        if requests_path.is_file():
            # On régénère les exemples pour cohérence (même seed que la 1ère exécution)
            rng = random.Random(SEED + batch_num)
            for code in codes_chunk:
                if code in code_examples_map:
                    continue
                code_examples_map[code] = get_cepidc_examples(code, cepidc_df, rng=rng)
        return result_path

    print(f"\n=== batch {batch_num} : {len(codes_chunk):,} codes ===")
    requests_path = build_batch_jsonl(codes_chunk, batch_num, cepidc_df, code_examples_map)

    with open(requests_path, "rb") as f:
        content = f.read()
    print(f"  upload ({len(content):,} bytes)...")
    uploaded = client.files.upload(
        file={"file_name": requests_path.name, "content": content},
        purpose="batch",
    )

    job = client.batch.jobs.create(
        endpoint="/v1/chat/completions",
        input_files=[uploaded.id],
        model=MODEL,
        timeout_hours=BATCH_TIMEOUT_HOURS,
        metadata={"batch_num": str(batch_num)},
    )
    print(f"  job.id={job.id}")

    last_print = 0.0
    last_status = None
    t0 = time.time()
    while True:
        job = client.batch.jobs.get(job_id=job.id)
        now = time.time()
        if job.status != last_status or now - last_print > POLL_VERBOSE_INTERVAL_S:
            elapsed_min = (now - t0) / 60
            print(
                f"    [{time.strftime('%H:%M:%S')}] status={job.status}  "
                f"completed={job.completed_requests}/{job.total_requests}  "
                f"({elapsed_min:.0f} min écoulées)"
            )
            last_status = job.status
            last_print = now
        if job.status in ("SUCCESS", "FAILED", "TIMEOUT_EXCEEDED", "CANCELLED"):
            break
        if now - t0 > BATCH_TIMEOUT_HOURS * 3600:
            raise RuntimeError(f"Timeout polling batch {batch_num} après {BATCH_TIMEOUT_HOURS}h")
        time.sleep(POLL_INTERVAL_S)

    if job.status != "SUCCESS":
        raise RuntimeError(
            f"Batch {batch_num} terminé avec status={job.status}, "
            f"failed_requests={job.failed_requests}"
        )

    print(f"  download → {result_path.name}")
    resp = client.files.download(file_id=job.output_file)
    if hasattr(resp, "iter_bytes"):
        with open(result_path, "wb") as f:
            for chunk in resp.iter_bytes():
                f.write(chunk)
    elif hasattr(resp, "read"):
        result_path.write_bytes(resp.read())
    else:
        result_path.write_bytes(bytes(resp))
    return result_path

In [ ]:
# === GÉNÉRATION MASSIVE (4 batches complets) ===
# ⚠ Coût ~50 € (Mistral Large batch -50 %)
# ⚠ Durée 4-96 h (typiquement <12 h)
#
# Reprise après interruption : ré-exécuter cette cellule. Les batches déjà
# terminés (fichier batch_<n>_results.jsonl présent) sont skippés.

code_to_cepidc_examples: dict[str, list[str]] = {}

# Charger un mapping existant si déjà sauvegardé (reprise)
if CEPIDC_EXAMPLES_PATH.is_file():
    with CEPIDC_EXAMPLES_PATH.open() as f:
        code_to_cepidc_examples = json.load(f)
    print(f"Mapping cepidc_examples chargé : {len(code_to_cepidc_examples):,} codes")

t0_total = time.time()
result_paths = []
for batch_num, chunk in enumerate(chunks, 1):
    path = run_batch(chunk, batch_num, cepidc, code_to_cepidc_examples)
    result_paths.append(path)
    # Sauvegarder le mapping après chaque batch (résistance aux interruptions)
    CEPIDC_EXAMPLES_PATH.parent.mkdir(parents=True, exist_ok=True)
    with CEPIDC_EXAMPLES_PATH.open("w", encoding="utf-8") as f:
        json.dump(code_to_cepidc_examples, f, ensure_ascii=False)

elapsed_h = (time.time() - t0_total) / 3600
print(f"\n=== 4 batches terminés en {elapsed_h:.1f}h ===")
print(f"Fichiers produits : {[p.name for p in result_paths]}")

In [ ]:
# Consolidation : agrège tous les résultats en un seul JSONL.
# Format identique à phase 1 (avec champ batch_num en plus).

# Recharger les mappings (au cas où les cellules 8/12 n'auraient pas été ré-exécutées)
with CEPIDC_EXAMPLES_PATH.open() as f:
    code_to_cepidc_examples = json.load(f)

results: list[dict] = []
errors_api: list[dict] = []
errors_parsing: list[dict] = []

for batch_num in range(1, len(chunks) + 1):
    path = BATCH_FILES_DIR / f"batch_{batch_num}_results.jsonl"
    if not path.is_file():
        print(f"⚠ batch {batch_num} : fichier {path.name} absent — skip")
        continue
    with path.open() as f:
        for line in f:
            rec = json.loads(line)
            custom_id = rec["custom_id"]
            if rec.get("error") or rec["response"]["status_code"] != 200:
                errors_api.append({
                    "code": custom_id,
                    "batch_num": batch_num,
                    "error": rec.get("error") or f"status {rec['response']['status_code']}",
                })
                continue
            content = rec["response"]["body"]["choices"][0]["message"]["content"]
            try:
                parsed = json.loads(content)
                formulations = parsed.get("formulations", []) if isinstance(parsed, dict) else []
            except json.JSONDecodeError as exc:
                errors_parsing.append({
                    "code": custom_id,
                    "batch_num": batch_num,
                    "parse_error": str(exc),
                    "raw_preview": content[:500],
                })
                continue
            usage = rec["response"]["body"]["usage"]
            cepidc_examples = code_to_cepidc_examples.get(custom_id, [])
            results.append({
                "code": custom_id,
                "libelle": code_to_libelle.get(custom_id, ""),
                "formulations": formulations,
                "n_formulations": len(formulations),
                "n_cepidc_examples": len(cepidc_examples),
                "cepidc_examples": cepidc_examples,
                "model": MODEL,
                "temperature": TEMPERATURE,
                "prompt_tokens": usage["prompt_tokens"],
                "completion_tokens": usage["completion_tokens"],
                "batch_num": batch_num,
            })

# Écriture du JSONL consolidé
CONSOLIDATED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with CONSOLIDATED_OUTPUT.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Consolidation : {len(results):,} lignes écrites dans {CONSOLIDATED_OUTPUT.name}")
print(f"  taille : {CONSOLIDATED_OUTPUT.stat().st_size / 1_000_000:.1f} Mo")
print(f"  erreurs API     : {len(errors_api)}")
print(f"  erreurs parsing : {len(errors_parsing)}")

In [ ]:
# === RÉCAP FINAL ===

print("=" * 70)
print("RÉCAP PHASE 2 — GÉNÉRATION MASSIVE")
print("=" * 70)

n_codes_attendus = len(codes_sorted)
n_codes_traites = len(results) + len(errors_api) + len(errors_parsing)
n_succes = len(results)

print(f"\nCodes attendus      : {n_codes_attendus:>6,}")
print(f"Codes traités       : {n_codes_traites:>6,}")
print(f"Succès              : {n_succes:>6,}  ({100*n_succes/n_codes_attendus:.1f}%)")
print(f"Erreurs API         : {len(errors_api):>6}")
print(f"Erreurs JSON parsing: {len(errors_parsing):>6}")

tot_in = sum(r["prompt_tokens"] for r in results)
tot_out = sum(r["completion_tokens"] for r in results)
cout = tot_in / 1_000_000 * 1 + tot_out / 1_000_000 * 3  # tarif batch -50%

print(f"\nTokens totaux       :")
print(f"  Input  : {tot_in/1_000_000:.2f} M")
print(f"  Output : {tot_out/1_000_000:.2f} M")
print(f"Coût estimé (batch) : ~{cout:.2f} €  (remise –50 % vs API standard)")

# Distribution succès par chapitre
print(f"\nDistribution succès par chapitre :")
if results:
    df = pl.DataFrame({"code": [r["code"] for r in results]}).join(
        index_df.select("code", "chapter"), on="code", how="left"
    )
    expected = index_df.group_by("chapter").len().rename({"len": "expected"})
    got = df.group_by("chapter").len().rename({"len": "got"})
    dist = expected.join(got, on="chapter", how="left").with_columns(
        pl.col("got").fill_null(0)
    ).with_columns(
        pl.col("chapter").replace_strict(ROMAN_TO_INT, default=99).alias("chap_num")
    ).sort("chap_num")
    for row in dist.iter_rows(named=True):
        print(f"  {row['chapter']:6s} : {row['got']:5,} / {row['expected']:5,}")

print(f"\nFichier consolidé : {CONSOLIDATED_OUTPUT}")
print(f"  taille : {CONSOLIDATED_OUTPUT.stat().st_size / 1_000_000:.1f} Mo")

if errors_api or errors_parsing:
    print(f"\nTop 10 erreurs :")
    for e in (errors_api + errors_parsing)[:10]:
        print(f"  {e['code']:8s} (batch {e['batch_num']}) : {e.get('error') or e.get('parse_error', 'JSON parse')[:80]}")